# Asset Price Modeling: Full Walkthrough

This notebook is an educational companion for the implementation in `quant_edu.asset_price_modeling`.

It combines theory, equations, and executable code to produce figures for all required models.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from quant_edu.asset_price_modeling import (
    estimate_drift_and_volatility,
    fetch_close_prices,
    log_returns_from_prices,
    realized_volatility,
    simulate_brownian_motion,
    simulate_geometric_brownian_motion,
    simulate_jump_diffusion,
    simulate_symmetric_random_walk,
)

plt.style.use("seaborn-v0_8")

## 1) Symmetric Random Walk

Let coin-toss outcomes produce increments:

$$
X_j = \begin{cases}
+1, & \text{with probability } \frac{1}{2} \\n-1, & \text{with probability } \frac{1}{2}
\end{cases}
$$

and define

$$
M_k = \sum_{j=1}^{k} X_j, \qquad M_0=0.
$$

The process moves up/down by one unit at each step.

In [ ]:
rw = simulate_symmetric_random_walk(n_steps=200, n_paths=6, seed=7)

fig, ax = plt.subplots(figsize=(10, 4))
for i in range(rw.shape[0]):
    ax.plot(rw[i], linewidth=1.4)
ax.set_title("Symmetric Random Walk Paths")
ax.set_xlabel("Step")
ax.set_ylabel("M_k")
ax.grid(True, linestyle="--", linewidth=0.5)
plt.show()


## 2) Brownian Motion

Brownian increments over a partition satisfy:

$$
W(t_{i+1}) - W(t_i) \sim \mathcal{N}(0, t_{i+1}-t_i).
$$

With uniform step size $\Delta t$, simulation uses $\mathcal{N}(0, \Delta t)$ increments.

In [ ]:
n_steps = 252
horizon = 1.0
time_axis = np.linspace(0.0, horizon, n_steps + 1)
bm = simulate_brownian_motion(n_steps=n_steps, horizon=horizon, n_paths=6, seed=42)

fig, ax = plt.subplots(figsize=(10, 4))
for i in range(bm.shape[0]):
    ax.plot(time_axis, bm[i], linewidth=1.4)
ax.set_title("Brownian Motion Paths")
ax.set_xlabel("Time")
ax.set_ylabel("W(t)")
ax.grid(True, linestyle="--", linewidth=0.5)
plt.show()


## 3) Geometric Brownian Motion (GBM)

GBM stochastic differential equation:

$$
\frac{dS(t)}{S(t)} = \alpha \, dt + \sigma \, dW(t).
$$

Closed-form path representation:

$$
S(t) = S(0)\exp\left(\sigma W(t)+\left(\alpha-\frac{1}{2}\sigma^2\right)t\right).
$$

In [ ]:
gbm = simulate_geometric_brownian_motion(
    n_steps=252,
    horizon=1.0,
    n_paths=6,
    s0=100.0,
    drift=0.08,
    volatility=0.25,
    seed=42,
)

fig, ax = plt.subplots(figsize=(10, 4))
for i in range(gbm.shape[0]):
    ax.plot(time_axis, gbm[i], linewidth=1.4)
ax.set_title("GBM Price Paths")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Price")
ax.grid(True, linestyle="--", linewidth=0.5)
plt.show()


## 4) Drift and Volatility Estimation

For one-step log return

$$
r_j = \log\left(\frac{S_{t_{j+1}}}{S_{t_j}}\right),
$$

GBM implies

$$
\mathbb{E}[r_j] = \left(\alpha - \frac{1}{2}\sigma^2\right)\Delta t,\qquad
\mathrm{Var}(r_j) = \sigma^2\Delta t.
$$

Realized volatility approximation:

$$
\sigma^2 \approx \frac{1}{T_2-T_1}\sum_{j}\left(\log\frac{S(t_{j+1})}{S(t_j)}\right)^2.
$$

This section estimates on real AAPL 2024 daily closes from the fixture dataset.

In [ ]:
symbol = "AAPL"
period = "1y"
dt = 1 / 252

sample_prices = fetch_close_prices(symbol, period=period)
estimate = estimate_drift_and_volatility(prices=sample_prices, dt=dt)
rv = realized_volatility(prices=sample_prices, dt=dt)
returns = log_returns_from_prices(sample_prices)

print(f"Symbol: {symbol}  |  {sample_prices.size} trading days loaded")
print("Estimated drift:     ", round(estimate.drift, 4))
print("Estimated volatility:", round(estimate.volatility, 4))
print("Realized volatility: ", round(rv, 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(
    np.arange(sample_prices.size),
    sample_prices,
    color="tab:blue",
    linewidth=2.0,
)
axes[0].set_title(f"Real {symbol} Daily Close Path (last {period})")
axes[0].set_xlabel("Trading day index")
axes[0].set_ylabel("Price")
axes[0].grid(True, linestyle="--", linewidth=0.5)

axes[1].hist(returns, bins=25, color="tab:green", alpha=0.8, edgecolor="black")
axes[1].set_title("Log-Return Distribution")
axes[1].set_xlabel("Log Return")
axes[1].set_ylabel("Frequency")
axes[1].grid(True, linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.show()

## 5) Jump Diffusion

A practical Merton-style discrete form:

$$
\Delta \log S \approx \left(\alpha-\frac{1}{2}\sigma^2-\lambda\kappa\right)\Delta t + \sigma\sqrt{\Delta t}Z + J_{\Delta t},
$$

where jump arrivals follow

$$
N_{\Delta t} \sim \mathrm{Poisson}(\lambda \Delta t).
$$

In [ ]:
jump = simulate_jump_diffusion(
    n_steps=252,
    horizon=1.0,
    n_paths=6,
    s0=100.0,
    drift=0.08,
    volatility=0.20,
    jump_intensity=6.0,
    jump_mean=-0.03,
    jump_std=0.10,
    seed=42,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for i in range(gbm.shape[0]):
    axes[0].plot(time_axis, gbm[i], linewidth=1.3)
for i in range(jump.shape[0]):
    axes[1].plot(time_axis, jump[i], linewidth=1.3)

axes[0].set_title("GBM (No Jump)")
axes[1].set_title("Jump Diffusion")
for ax in axes:
    ax.set_xlabel("Time (years)")
    ax.grid(True, linestyle="--", linewidth=0.5)
axes[0].set_ylabel("Price")
plt.tight_layout()
plt.show()


## Reproducible Full Script

A script version of this complete workflow is provided at:

- `examples/asset_price_modeling/full_modeling_showcase.py`

Run with:

```bash
python examples/asset_price_modeling/full_modeling_showcase.py
```